## Tranco Top Sites Data Acquisition

Source: https://tranco-list.eu/

Each row represents one domain and its popularity rank in the selected Tranco snapshot.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import polars as pl

In [2]:
# download the Tranco snapshot
raw_dir = Path("../datasets/tranco/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
csv_path = raw_dir / "tranco_Y8YJG.csv"

if not csv_path.exists():
    urlretrieve("https://tranco-list.eu/download/Y8YJG/1000000", csv_path)

In [3]:
# load the dataframe
df = pl.read_csv(csv_path, has_header=False,
                 schema={"rank": pl.Int64, "domain": pl.String})

In [4]:
df.head()

rank,domain
i64,str
1,"""google.com"""
2,"""cloudflare.com"""
3,"""facebook.com"""
4,"""gstatic.com"""
5,"""googleapis.com"""


In [5]:
df.shape

(1000000, 2)

In [6]:
df.columns

['rank', 'domain']

In [7]:
df.schema

Schema([('rank', Int64), ('domain', String)])

In [8]:
df.null_count()

rank,domain
u32,u32
0,0


In [9]:
df.height - df["domain"].n_unique()

0

Tranco contains pay-level domains, so dot count should not be treated as a subdomain count.

In [10]:
# add domain structure features
df = df.with_columns(
    pl.col("domain").str.len_chars().alias("domain_length"),
    pl.col("domain").str.count_matches(r"\.").alias("number_of_dots"),
    pl.col("domain").str.contains("-", literal=True).alias("contains_hyphen"),
    pl.col("domain").str.contains(r"[0-9]").alias("contains_number"),
)

In [11]:
df.head()

rank,domain,domain_length,number_of_dots,contains_hyphen,contains_number
i64,str,u32,u32,bool,bool
1,"""google.com""",10,1,false,false
2,"""cloudflare.com""",14,1,false,false
3,"""facebook.com""",12,1,false,false
4,"""gstatic.com""",11,1,false,false
5,"""googleapis.com""",14,1,false,false


In [12]:
df.shape

(1000000, 6)

In [13]:
df.columns

['rank',
 'domain',
 'domain_length',
 'number_of_dots',
 'contains_hyphen',
 'contains_number']

In [14]:
df.schema

Schema([('rank', Int64),
        ('domain', String),
        ('domain_length', UInt32),
        ('number_of_dots', UInt32),
        ('contains_hyphen', Boolean),
        ('contains_number', Boolean)])

In [15]:
df.null_count()

rank,domain,domain_length,number_of_dots,contains_hyphen,contains_number
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0
